# Film Junky Union — Clasificación de sentimientos de reseñas

**Proyecto de NLP / Machine Learning supervisado**

Objetivo: clasificar reseñas de películas como positivas o negativas y evaluar el desempeño mediante F1. El objetivo mínimo indicado por el proyecto es **F1 ≥ 0.85**.

Este notebook contiene un baseline reproducible con TF-IDF + Regresión Logística. Los experimentos incompletos de spaCy, LightGBM y BERT del notebook académico original se excluyeron de esta versión pública para evitar presentar código parcial como resultados terminados.

## 1. Configuración e importaciones

Ejecuta las celdas en orden. El dataset `imdb_reviews.tsv` debe obtenerse por un canal autorizado y colocarse en la raíz del repositorio. No se incluye por restricciones de distribución.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, average_precision_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

RANDOM_STATE = 12345
DATA_PATH = Path('imdb_reviews.tsv')


## 2. Carga y validación inicial de datos

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'No se encontró {DATA_PATH}. Coloca imdb_reviews.tsv en la raíz del proyecto.'
    )

df_reviews = pd.read_csv(DATA_PATH, sep='\t', dtype={'votes': 'Int64'})
print(f'Dimensiones: {df_reviews.shape}')
display(df_reviews.head())
display(df_reviews.isna().sum().sort_values(ascending=False).head(10))
print('Filas completamente duplicadas:', df_reviews.duplicated().sum())

required_columns = {'review', 'pos', 'ds_part'}
missing_columns = required_columns - set(df_reviews.columns)
if missing_columns:
    raise ValueError(f'Faltan columnas requeridas: {missing_columns}')
if df_reviews['review'].isna().any() or df_reviews['pos'].isna().any():
    raise ValueError('Hay valores nulos en review o pos; revisa el dataset antes de entrenar.')


## 3. Análisis exploratorio de la variable objetivo

In [ ]:
display(df_reviews['pos'].value_counts(dropna=False).rename('cantidad').to_frame())
ax = df_reviews['pos'].value_counts(normalize=True).sort_index().plot(
    kind='bar', title='Distribución de la variable objetivo', rot=0
)
ax.set_xlabel('Etiqueta pos')
ax.set_ylabel('Proporción')
plt.tight_layout()
plt.show()
print('Particiones disponibles:', df_reviews['ds_part'].value_counts(dropna=False).to_dict())


## 4. Preparación del texto y partición train/test

Se conserva la partición predefinida `ds_part` del conjunto de datos para mantener la separación original. La normalización elimina caracteres que no sean letras inglesas o apóstrofos, siguiendo el enfoque del notebook académico.

In [ ]:
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z']", ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df_reviews = df_reviews.copy()
df_reviews['review_norm'] = df_reviews['review'].astype(str).map(clean_text)
train_df = df_reviews.loc[df_reviews['ds_part'].eq('train')].copy()
test_df = df_reviews.loc[df_reviews['ds_part'].eq('test')].copy()
if train_df.empty or test_df.empty:
    raise ValueError('La columna ds_part debe contener las particiones train y test.')
X_train, y_train = train_df['review_norm'], train_df['pos'].astype(int)
X_test, y_test = test_df['review_norm'], test_df['pos'].astype(int)
print('Train:', X_train.shape, 'Test:', X_test.shape)


## 5. Modelo de referencia (Dummy Classifier)

In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train.to_frame(), y_train)
dummy_pred = dummy.predict(X_test.to_frame())
print('Dummy accuracy:', round(accuracy_score(y_test, dummy_pred), 4))
print('Dummy F1:', round(f1_score(y_test, dummy_pred, zero_division=0), 4))


## 6. Modelo principal: TF-IDF + Regresión Logística

El vectorizador se ajusta exclusivamente con el conjunto de entrenamiento. El uso de `Pipeline` mantiene el preprocesamiento y el modelo unidos y reduce el riesgo de fuga de información.

In [ ]:
model = Pipeline(steps=[
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=10_000)),
    ('classifier', LogisticRegression(random_state=RANDOM_STATE, max_iter=1_000))
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]


## 7. Evaluación del modelo

In [ ]:
results = {
    'Accuracy': accuracy_score(y_test, predictions),
    'F1': f1_score(y_test, predictions),
    'Average Precision': average_precision_score(y_test, probabilities),
    'ROC AUC': roc_auc_score(y_test, probabilities),
}
display(pd.DataFrame([results], index=['TF-IDF + Logistic Regression']).round(4))
print(classification_report(y_test, predictions, digits=4))


In [ ]:
cm = confusion_matrix(y_test, predictions)
cm_display = ConfusionMatrixDisplay(confusion_matrix=cm)
cm_display.plot(cmap='Blues', values_format='d')
plt.title('Matriz de confusión — conjunto de prueba')
plt.tight_layout()
plt.show()


## 8. Revisión de errores

Se muestran ejemplos de falsos positivos y falsos negativos para inspeccionar casos donde el modelo no coincide con la etiqueta real. Revisa manualmente los ejemplos antes de extraer conclusiones sobre patrones lingüísticos.

In [ ]:
error_analysis = test_df[['review', 'pos']].copy()
error_analysis['prediction'] = predictions
error_analysis['probability_positive'] = probabilities
false_positives = error_analysis.query('pos == 0 and prediction == 1')
false_negatives = error_analysis.query('pos == 1 and prediction == 0')
print('Falsos positivos:', len(false_positives))
print('Falsos negativos:', len(false_negatives))
display(false_positives.head(5))
display(false_negatives.head(5))


## 9. Conclusiones

Completa esta sección después de ejecutar el notebook:

- **F1 en test:** anota el valor calculado en la sección 7.
- **Objetivo F1 ≥ 0.85:** indica si se alcanzó con la ejecución actual.
- **Comparación con Dummy:** explica cuánto mejora el modelo principal frente a la línea base.
- **Errores:** describe patrones observados en falsos positivos y falsos negativos.
- **Limitaciones:** el modelo TF-IDF no comprende plenamente sarcasmo, contexto, ironía ni dependencias semánticas complejas. La partición de evaluación y las etiquetas dependen del dataset original.

No copies métricas históricas del notebook anterior sin verificar que se reproducen en esta versión.

## 10. Reproducibilidad y siguientes pasos

1. Ejecuta todas las celdas desde un kernel limpio.
2. Confirma que las métricas se generan sin errores.
3. Registra versiones de Python y librerías si deseas fijar el entorno.
4. Actualiza el README con resultados reproducidos.
5. No publiques el dataset si no tienes autorización para redistribuirlo.